In [12]:
import pandas as pd
import json
import re

import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [5]:
# 1. Cargar datos
partidos = pd.read_csv('../data/partidos_rankeados.csv')
with open('../data/jugadores_todos_los_paises.json', 'r', encoding='utf-8') as f:
    jugadores_dict = json.load(f)

# 2. Función para limpiar y transformar el valor de mercado
def clean_market_value(value_str):
    if pd.isna(value_str) or not isinstance(value_str, str):
        return 0.0
    # Remover símbolo de euro y espacios
    val = value_str.replace('€', '').strip()
    if 'm' in val:
        return float(val.replace('m', ''))
    elif 'k' in val:
        return float(val.replace('k', '')) / 1000.0
    return 0.0

In [15]:
# 3. Calcular valor total por país
valores_paises = {}
for pais, lista_jugadores in jugadores_dict.items():
    total_val = sum(clean_market_value(j.get('Market Value', '0')) for j in lista_jugadores)
    valores_paises[pais] = total_val

# 4. Mapear valores a la matriz de partidos
partidos['home_value'] = partidos['home_team'].map(valores_paises).fillna(0.0)
partidos['away_value'] = partidos['away_team'].map(valores_paises).fillna(0.0)

# 5. Ingeniería de Características (Features del Partido)
matriz_partidos = pd.DataFrame()
matriz_partidos['id_partido'] = partidos['id']
matriz_partidos['home_team'] = partidos['home_team']
matriz_partidos['away_team'] = partidos['away_team']

#Features de Fecha
partidos['date'] = pd.to_datetime(partidos['date'])
# OPCIÓN A: Día en formato texto (ej: 'Thursday', 'Friday')
partidos['day_of_week_name'] = partidos['date'].dt.day_name()
# OPCIÓN B: Día en formato numérico (Lunes = 0, Domingo = 6)
matriz_partidos['day_of_week_num'] = partidos['date'].dt.dayofweek
# Crea una columna que vale 1 si es Sábado (5) o Domingo (6), y 0 si es día de semana
matriz_partidos['is_weekend'] = partidos['date'].dt.dayofweek.isin([5, 6]).astype(int)

# Features numéricas puras
matriz_partidos['ELO_match_rating'] = partidos['match_rating']
matriz_partidos['ELO_diff'] = (partidos['home_elo_rating'] - partidos['away_elo_rating']).abs()
matriz_partidos['home_value'] = partidos['home_value']
matriz_partidos['away_value'] = partidos['away_value']
matriz_partidos['match_value'] = partidos['home_value'] + partidos['away_value']
# 1. Aplicar logaritmo natural para achatar la diferencia entre ricos y pobres
#matriz_partidos['match_value_log'] = np.log1p(matriz_partidos['match_value'])

# 2. Ahora sí, aplicar MinMaxScaler a la columna con logaritmo
scaler = MinMaxScaler()
#matriz_partidos['match_value_scaled'] = scaler.fit_transform(matriz_partidos[['match_value_log']])
matriz_partidos['match_value_diff'] = (partidos['home_value'] - partidos['away_value']).abs()
# 1. Aplicar logaritmo para suavizar las diferencias extremas
#matriz_partidos['match_value_diff_log'] = np.log1p(matriz_partidos['match_value_diff'])

# 2. Escalar al rango [0, 1] con tu scaler previamente definido
#matriz_partidos['match_value_diff_scaled'] = scaler.fit_transform(matriz_partidos[['match_value_diff_log']])
matriz_partidos['match_hora_utc'] = pd.to_datetime(partidos['time_utc'], format='%H:%M:%S').dt.hour

# Guardar o inspeccionar la matriz resultante
matriz_partidos

,id_partido,home_team,away_team,day_of_week_num,is_weekend,ELO_match_rating,ELO_diff,home_value,away_value,match_value,match_value_diff,match_hora_utc
0,2,Mexico,South Africa,3,0,3382,334,85.600,52.700,138.300,32.900,19
1,1,Korea Republic,Czechia,4,0,3478,26,143.400,196.425,339.825,53.025,2
2,7,Canada,Bosnia-Herzegovina,4,0,3378,190,222.450,140.400,362.850,82.050,19
3,20,USA,Paraguay,5,1,3554,112,356.700,139.000,495.700,217.700,1
4,9,Qatar,Switzerland,5,1,3314,464,20.475,317.600,338.075,297.125,19
...,...,...,...,...,...,...,...,...,...,...,...,...
67,72,Croatia,Ghana,5,1,3435,425,356.300,295.175,651.475,61.125,21
68,65,Congo DR,Uzbekistan,5,1,2934,520,147.250,78.725,225.975,68.525,23
69,66,Colombia,Portugal,5,1,3959,9,303.950,965.000,1268.950,661.050,23
70,57,Algeria,Austria,6,1,3570,84,227.800,259.700,487.500,31.900,2


In [16]:
matriz_partidos = matriz_partidos.sort_values(by='id_partido', ascending=True)
matriz_partidos.to_csv('../data/matriz_partidos.csv', index=False, encoding="utf-8")